# Notebook 5: generic-text pairs for the `kd_generic` arm
**Kaggle: GPU T4, Internet ON, Secret `HF_TOKEN`, attach Datasets `nli-data` and `nli-llama-adapter` (folder `llama1b_lora_adapter`).**

1. Premises: sentences from three generic corpora (Wikipedia / news / movie reviews). None is an NLI dataset and none is MNLI.
2. Hypotheses: the UNTUNED Llama-3.2-1B-Instruct writes one hypothesis per premise (asked to be entailed / contradicted / neutral,
   in equal parts, only to get class diversity; these requests are NOT used as labels).
3. Labels: the fine-tuned teacher (Notebook 3 adapter) gives logits for every pair. No gold labels exist for these pairs.
4. Decontamination: premises that share text with SNLI or MNLI (exact match or any 8-word sequence) are removed BEFORE generation.

Output: `generic_pairs_<N>.csv` (premise, hypothesis, z_ent, z_neu, z_con, source, requested) + `generic_stats.json`.
Expected runtime: about 45 minutes for 60K pairs. Look at the printed sample pairs: they are the quality check.

In [1]:
import glob, json, os, random, re, subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)   # Kaggle's old torchao breaks peft
from pathlib import Path
import numpy as np, pandas as pd, torch
try:
    import peft
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True); import peft
from peft import PeftModel
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def find(pattern):
    h = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/" + pattern, recursive=True)
    assert h, f"not found: {pattern} (is the right Dataset attached?)"
    return h[0]

ROOT = Path(find("nli_common.py")).parent; sys.path.insert(0, str(ROOT)); DATA = str(ROOT / "data")
from nli_common import load_split
ADAPTER = str(Path(find("llama1b_lora_adapter/adapter_config.json")).parent)
OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/generic")); OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "turn on the GPU"
dev = "cuda"; SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Kaggle Secret HF_TOKEN"

MODEL = "meta-llama/Llama-3.2-1B-Instruct"
N_PER_SOURCE = int(os.environ.get("N_PER_SOURCE", 20_000))      # 3 sources -> ~60K pairs
GEN_BS, MAX_NEW = 48, 40
print("GPU:", torch.cuda.get_device_name(0), "| pairs target:", 3 * N_PER_SOURCE)

GPU: Tesla T4 | pairs target: 60000


## 1. Premises from generic corpora

In [2]:
def clean_text(t):
    t = t.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".").replace("<br />", " ").replace("\n", " ")
    t = re.sub(r"\s+([.,;:!?%)])", r"\1", t); t = re.sub(r"([(])\s+", r"\1", t); t = re.sub(r"\s+", " ", t).strip()
    return t

SENT_SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z"\'])')
def premises_from_paragraphs(paras, n_target, rng):
    out = []
    for para in paras:
        sents = [s for s in SENT_SPLIT.split(clean_text(para)) if s]
        i = 0
        while i < len(sents):
            s = sents[i]; w = len(s.split())
            if 8 <= w <= 45 and s.isascii() and sum(c.isdigit() for c in s) < 0.15 * len(s):
                if i + 1 < len(sents) and rng.random() < 0.25:                      # sometimes a 2-sentence premise
                    s2 = s + " " + sents[i + 1]
                    if len(s2.split()) <= 60 and s2.isascii(): s, i = s2, i + 1
                out.append(s)
            i += 1
        if len(out) >= n_target * 3: break
    return out

def load_sources():
    rng = random.Random(SEED); src = {}
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
        idx = rng.sample(range(len(ds)), 200_000)
        paras = ds.select(idx)["text"]; paras = [p for p in paras if len(p) > 200 and not p.strip().startswith("=")]
        src["wikipedia"] = premises_from_paragraphs(paras, N_PER_SOURCE * 2, rng)
    except Exception as e: print("wikipedia failed:", e)
    try:
        ds = load_dataset("fancyzhx/ag_news", split="train"); paras = ds.select(rng.sample(range(len(ds)), 80_000))["text"]
        src["news"] = premises_from_paragraphs(paras, N_PER_SOURCE * 2, rng)
    except Exception as e: print("news failed:", e)
    try:
        ds = load_dataset("stanfordnlp/imdb", split="train"); paras = ds.select(rng.sample(range(len(ds)), 25_000))["text"]
        src["reviews"] = premises_from_paragraphs(paras, N_PER_SOURCE * 2, rng)
    except Exception as e: print("reviews failed:", e)
    return src

src = load_sources()
for k, v in src.items(): print(f"{k:<10} candidate premises: {len(v):,}")
assert src, "no corpus could be loaded: check Internet is ON"

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00001-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

... (20 outputs left out)

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

wikipedia  candidate premises: 120,003
news       candidate premises: 87,144
reviews    candidate premises: 120,001


## 2. Decontamination against SNLI and MNLI (done before any generation)

In [3]:
def norm(t): return re.sub(r"[^a-z0-9 ]", "", t.lower()).strip()
def ngrams(t, n=8):
    w = norm(t).split(); return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}

ref_sets = [load_split(n, DATA) for n in ["snli_val", "snli_test", "mnli_m", "mnli_mm", "train_100k"]]
ref_exact, ref_ngrams = set(), set()
for d in ref_sets:
    for col in ("premise", "hypothesis"):
        for t in d[col]:
            ref_exact.add(norm(t)); ref_ngrams |= ngrams(t)
print(f"reference texts: {len(ref_exact):,} exact, {len(ref_ngrams):,} 8-grams")

kept, removed = {}, {}
for k, plist in src.items():
    seen, good = set(), []
    for p in plist:
        n = norm(p)
        if n in seen: continue
        seen.add(n)
        if n in ref_exact or (ngrams(p) & ref_ngrams): continue
        good.append(p)
    removed[k] = len(plist) - len(good); kept[k] = good
    print(f"{k:<10} kept {len(good):,} / {len(plist):,} (removed {removed[k]:,})")
rng = random.Random(SEED)
premises = [(p, k) for k, v in kept.items() for p in rng.sample(v, min(N_PER_SOURCE, len(v)))]
rng.shuffle(premises)
print("premises selected:", len(premises), pd.Series([k for _, k in premises]).value_counts().to_dict())

reference texts: 213,939 exact, 741,398 8-grams
wikipedia  kept 119,839 / 120,003 (removed 164)
news       kept 86,694 / 87,144 (removed 450)
reviews    kept 119,728 / 120,001 (removed 273)
premises selected: 60000 {'news': 20000, 'reviews': 20000, 'wikipedia': 20000}


## 3. Hypotheses from the untuned Llama

In [4]:
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
if "<|finetune_right_pad_id|>" in tok.get_vocab(): tok.pad_token = "<|finetune_right_pad_id|>"
elif tok.pad_token is None: tok.pad_token = tok.eos_token
PAD = tok.pad_token_id
try:
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, token=HF_TOKEN)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, token=HF_TOKEN)
base.to(dev).eval()

REQ = {0: "Write one short sentence that is definitely true if the text is true.",
       1: "Write one short sentence about the same situation that may or may not be true, because the text does not say.",
       2: "Write one short sentence that cannot be true if the text is true (it contradicts the text)."}
def gen_prompt(premise, rel):
    msg = [{"role": "user", "content": f"Text: {premise}\n\nTask: {REQ[rel]}\nReply with only that one sentence."}]
    return tok.apply_chat_template(msg, add_generation_prompt=True, tokenize=False)

def clean_hyp(t):
    t = t.strip().split("\n")[0].strip().strip('"\u201c\u201d\'').strip()
    return re.sub(r"^(sentence|hypothesis|answer)\s*:\s*", "", t, flags=re.I)

@torch.no_grad()
def generate_all(prem_list):
    rels = [i % 3 for i in range(len(prem_list))]; random.Random(SEED).shuffle(rels)
    prompts = [gen_prompt(p, r) for (p, _), r in zip(prem_list, rels)]
    enc = tok(prompts, add_special_tokens=False).input_ids
    order = np.argsort([len(e) for e in enc]); hyps = [None] * len(prompts); t0 = time.time(); tok.padding_side = "left"
    for b, s in enumerate(range(0, len(order), GEN_BS)):
        idx = order[s:s + GEN_BS]; L = max(len(enc[j]) for j in idx)
        ids = torch.full((len(idx), L), PAD); mask = torch.zeros((len(idx), L), dtype=torch.long)
        for i, j in enumerate(idx): ids[i, L - len(enc[j]):] = torch.tensor(enc[j]); mask[i, L - len(enc[j]):] = 1
        out = base.generate(input_ids=ids.to(dev), attention_mask=mask.to(dev), max_new_tokens=MAX_NEW, do_sample=True,
                            temperature=0.7, top_p=0.9, pad_token_id=PAD)
        for i, j in enumerate(idx): hyps[j] = clean_hyp(tok.decode(out[i, L:], skip_special_tokens=True))
        if (b + 1) % 50 == 0:
            done = min(s + GEN_BS, len(order)); el = time.time() - t0
            print(f"  generated {done:,}/{len(order):,}  {done / el:.0f} ex/s  ETA {(len(order) - done) / (done / el) / 60:.1f} min")
    return hyps, rels

hyps, rels = generate_all(premises)
df = pd.DataFrame({"premise": [p for p, _ in premises], "source": [k for _, k in premises], "hypothesis": hyps, "requested": rels})

def ok_hyp(h, p):
    w = h.split()
    return 3 <= len(w) <= 30 and h[-1] in ".!?" and h.lower() != p.lower() and not h.lower().startswith(("here", "sure", "i ", "note", "this ")) and h.isascii()
n0 = len(df); df = df[[ok_hyp(h, p) if isinstance(h, str) and h else False for h, p in zip(df["hypothesis"], df["premise"])]].reset_index(drop=True)
print(f"hypotheses kept after filtering: {len(df):,} / {n0:,}")
print(df.sample(12, random_state=SEED)[["requested", "premise", "hypothesis"]].to_string(max_colwidth=90))

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  generated 2,400/60,000  24 ex/s  ETA 39.2 min
  generated 4,800/60,000  23 ex/s  ETA 40.3 min
  generated 7,200/60,000  22 ex/s  ETA 40.3 min
  generated 9,600/60,000  21 ex/s  ETA 39.8 min
  generated 12,000/60,000  21 ex/s  ETA 38.6 min
  generated 14,400/60,000  20 ex/s  ETA 37.2 min
  generated 16,800/60,000  20 ex/s  ETA 35.7 min
  generated 19,200/60,000  20 ex/s  ETA 34.1 min
  generated 21,600/60,000  20 ex/s  ETA 32.4 min
  generated 24,000/60,000  20 ex/s  ETA 30.7 min
  generated 26,400/60,000  19 ex/s  ETA 28.9 min
  generated 28,800/60,000  19 ex/s  ETA 27.0 min
  generated 31,200/60,000  19 ex/s  ETA 25.1 min
  generated 33,600/60,000  19 ex/s  ETA 23.2 min
  generated 36,000/60,000  19 ex/s  ETA 21.2 min
  generated 38,400/60,000  19 ex/s  ETA 19.2 min
  generated 40,800/60,000  19 ex/s  ETA 17.2 min
  generated 43,200/60,000  19 ex/s  ETA 15.1 min
  generated 45,600/60,000  18 ex/s  ETA 13.1 min
  generated 48,000/60,000  18 ex/s  ETA 11.0 min
  generated 50,400/60,00

## 4. Label with the fine-tuned teacher

In [5]:
model = PeftModel.from_pretrained(base, ADAPTER, is_trainable=False); model.eval()
LABEL_IDS = [tok(w, add_special_tokens=False).input_ids[0] for w in [" entailment", " neutral", " contradiction"]]
assert LABEL_IDS == [87092, 21277, 50859], LABEL_IDS
LABEL_IDS_T = torch.tensor(LABEL_IDS, device=dev)

def build_prompt(premise, hypothesis):      # MUST match Notebooks 3 and 4
    return (f"Premise: {premise}\nHypothesis: {hypothesis}\n"
            "Does the premise entail the hypothesis, contradict it, or is it neutral?\n"
            "Answer with one word: entailment, neutral, or contradiction.\nAnswer:")

def pad_left(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs): ids[i, L - len(s):] = torch.tensor(s); mask[i, L - len(s):] = 1
    return ids.to(dev), mask.to(dev), (mask.cumsum(1) - 1).clamp(min=0).to(dev)

_KEEP = {"name": "logits_to_keep"}
def last_logits(m, ids, mask, pos):
    while True:
        try:
            kw = {_KEEP["name"]: 1} if _KEEP["name"] else {}
            return m(input_ids=ids, attention_mask=mask, position_ids=pos, **kw).logits[:, -1, :]
        except TypeError:
            if _KEEP["name"] is None: raise
            _KEEP["name"] = "num_logits_to_keep" if _KEEP["name"] == "logits_to_keep" else None

@torch.no_grad()
def label_logits(d, bs=64):
    seqs = tok([build_prompt(p, h) for p, h in zip(d["premise"], d["hypothesis"])], add_special_tokens=True).input_ids
    order = np.argsort([len(s) for s in seqs]); out = np.zeros((len(d), 3), np.float32); t0 = time.time()
    for b, s in enumerate(range(0, len(order), bs)):
        idx = order[s:s + bs]; ids, mask, pos = pad_left([seqs[j] for j in idx])
        with torch.autocast("cuda", dtype=torch.float16): lg = last_logits(model, ids, mask, pos)
        out[idx] = lg[:, LABEL_IDS_T].float().cpu().numpy()
        if (b + 1) % 100 == 0:
            done = min(s + bs, len(order)); el = time.time() - t0
            print(f"  labelled {done:,}/{len(order):,}  {done / el:.0f} ex/s  ETA {(len(order) - done) / (done / el) / 60:.1f} min")
    return out

chk = load_split("snli_val", DATA).sample(2000, random_state=SEED)       # sanity: adapter must reproduce ~92% on SNLI val
acc = float((label_logits(chk).argmax(1) == chk["label"].to_numpy()).mean())
print(f"teacher accuracy on 2,000 SNLI-val rows: {acc:.4f} (full val: 0.9198)"); assert acc > 0.89, "adapter not loaded correctly"

z = label_logits(df)
df[["z_ent", "z_neu", "z_con"]] = z
p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)

teacher accuracy on 2,000 SNLI-val rows: 0.9275 (full val: 0.9198)
  labelled 6,400/53,993  71 ex/s  ETA 11.2 min
  labelled 12,800/53,993  67 ex/s  ETA 10.3 min
  labelled 19,200/53,993  64 ex/s  ETA 9.1 min
  labelled 25,600/53,993  61 ex/s  ETA 7.7 min
  labelled 32,000/53,993  59 ex/s  ETA 6.2 min
  labelled 38,400/53,993  57 ex/s  ETA 4.5 min
  labelled 44,800/53,993  56 ex/s  ETA 2.7 min
  labelled 51,200/53,993  54 ex/s  ETA 0.9 min


## 5. Diagnostics and save

In [6]:
pred = z.argmax(1)
conf = pd.crosstab(df["requested"].map({0: "asked: entail", 1: "asked: neutral", 2: "asked: contradict"}),
                   pd.Series(pred, name="teacher label").map({0: "entailment", 1: "neutral", 2: "contradiction"}))
print(conf, "\n")
stats = {"n_pairs": len(df), "per_source": df["source"].value_counts().to_dict(), "removed_by_decontamination": removed,
         "teacher_label_dist": {l: float((pred == i).mean()) for i, l in enumerate(["entailment", "neutral", "contradiction"])},
         "agreement_requested_vs_teacher": float((pred == df["requested"].to_numpy()).mean()),
         "mean_max_prob": float(p.max(1).mean()), "frac_max_prob_gt_0.9": float((p.max(1) > 0.9).mean()),
         "teacher_check_acc_snli_val_2k": acc, "premise_words_mean": float(df["premise"].str.split().str.len().mean()),
         "hyp_words_mean": float(df["hypothesis"].str.split().str.len().mean())}
print(json.dumps(stats, indent=2))
df.insert(0, "id", np.arange(len(df)))
fn = OUT / f"generic_pairs_{len(df) // 1000}k.csv"; df.to_csv(fn, index=False)
json.dump(stats, open(OUT / "generic_stats.json", "w"), indent=2)
print("saved:", sorted(os.listdir(OUT)))
print("Next: Save Version -> Save & Run All; make a Dataset 'nli-generic' from the output; Notebook 6 with RUN_ARMS=['kd_generic'].")

teacher label      contradiction  entailment  neutral
requested                                            
asked: contradict          12415         978     4952
asked: entail               1329       10779     6871
asked: neutral              1565        3120    11984 

{
  "n_pairs": 53993,
  "per_source": {
    "news": 18650,
    "wikipedia": 18093,
    "reviews": 17250
  },
  "removed_by_decontamination": {
    "wikipedia": 164,
    "news": 450,
    "reviews": 273
  },
  "teacher_label_dist": {
    "entailment": 0.2755357175930213,
    "neutral": 0.4409275276424722,
    "contradiction": 0.2835367547645065
  },
  "agreement_requested_vs_teacher": 0.65152890189469,
  "mean_max_prob": 0.7818494439125061,
  "frac_max_prob_gt_0.9": 0.33671031430000187,
  "teacher_check_acc_snli_val_2k": 0.9275,
  "premise_words_mean": 26.223732706091532,
  "hyp_words_mean": 16.233511751523345
}
saved: ['generic_pairs_53k.csv', 'generic_stats.json']
Next: Save Version -> Save & Run All; make a Dataset 'n